# Task 2 – Bias measurement on the original labels (before mitigation)

For each dataset we compute three metrics **directly on the ground-truth labels**, before any bias mitigation and before training any classifier:

* **DP_Diff** – demographic parity difference: P(Y=1 | unprivileged) − P(Y=1 | privileged). 0 is fair, negative means the unprivileged group gets the favorable label less often.
* **DIR** – disparate impact ratio: P(Y=1 | unprivileged) / P(Y=1 | privileged). 1 is fair; below 0.8 violates the EEOC 80% rule.
* **Consistency** – individual fairness (Zemel et al., 2013): 1 − mean |yᵢ − mean label of the k nearest neighbours of i| (k = 5). Close to 1 means similar people get similar labels.

### Datasets, versions and protected groups

| Dataset | File used | Favorable label (Y=1) | Protected attribute (privileged / unprivileged) |
|---|---|---|---|
| **Adult** | `adult.csv` | income > 50K | **sex** (Male / Female), **race** (White / non-White) |
| **German Credit** | `german.data` (original, categorical version) | good credit risk | **age** (≥ 25 / < 25), **sex** (male / female, derived from *personal status and sex*) |
| **COMPAS** | `cox-violent-parsed_filt.csv` | no recidivism (`is_recid` = 0) | **race** (Caucasian / non-Caucasian), **sex** (Female / Male) |

Why these versions:
* **German:** `german.data` keeps the readable attributes (age, personal status/sex), so the protected groups can be built directly. `german.data-numeric` is the same data re-coded without a header, with the sex information mixed into indicator columns, so it is not needed here.
* **COMPAS:** `compas-scores-raw.csv` is a long file with several COMPAS scales per assessment and no ground-truth outcome, so it is not suitable. `cox-violent-parsed.csv` and `cox-violent-parsed_filt.csv` contain the same rows; the *_filt* file only drops redundant columns, so we use it. It has several rows per person (survival intervals), which we reduce to one row per person.

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

DATA_DIR = "data"       # folder that contains the three data files
K = 5                   # neighbours for consistency
pd.set_option("display.width", 200)

## Metrics

In [2]:
def dp_diff(y, priv):
    """P(Y=1 | unprivileged) - P(Y=1 | privileged). y: 0/1 array, priv: 1 = privileged, 0 = unprivileged."""
    return y[priv == 0].mean() - y[priv == 1].mean()

def dir_ratio(y, priv):
    """P(Y=1 | unprivileged) / P(Y=1 | privileged)."""
    return y[priv == 0].mean() / y[priv == 1].mean()

def consistency(X, y, k=K):
    """Zemel et al. consistency: 1 - mean_i | y_i - mean(y of the k nearest neighbours of i) |."""
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float)
    idx = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X, return_distance=False)
    keep = idx != np.arange(len(X))[:, None]                    # drop the point itself (also with exact duplicates)
    order = np.argsort(~keep, axis=1, kind="stable")[:, :k]
    nbr = np.take_along_axis(idx, order, axis=1)
    return 1 - np.abs(y - y[nbr].mean(axis=1)).mean()

def make_features(df):
    """One-hot encode categorical columns, min-max scale numeric ones (as in Feldman et al.)."""
    df = df.copy()
    num = df.select_dtypes(include="number").columns
    df[num] = (df[num] - df[num].min()) / (df[num].max() - df[num].min()).replace(0, 1)
    return pd.get_dummies(df, dtype=float)

## Loading the data

Each dataset is read with a plain `pd.read_csv`. The loaders return the features (protected attributes removed), the 0/1 label (1 = favorable) and the protected attributes (1 = privileged, 0 = unprivileged).

In [3]:
# ---- Simple data loading: plain pd.read_csv for every dataset ----
# Each loader returns: features X (protected attributes removed), label y (1 = favorable outcome),
# and a dict of protected attributes (1 = privileged group, 0 = unprivileged group).

def load_adult():
    df = pd.read_csv(f"{DATA_DIR}/adult.csv")
    y = (df["income"] == ">50K").astype(int)                      # favorable: income > 50K
    prot = {"sex": (df["sex"] == "Male").astype(int),
            "race": (df["race"] == "White").astype(int)}
    X = df.drop(columns=["income", "sex", "race", "fnlwgt", "education"])
    return X, y, prot

def load_german():
    cols = ["checking_status", "duration", "credit_history", "purpose", "credit_amount", "savings",
            "employment_since", "installment_rate", "personal_status_sex", "other_debtors",
            "residence_since", "property", "age", "other_installment_plans", "housing",
            "existing_credits", "job", "num_liable", "telephone", "foreign_worker", "target"]
    df = pd.read_csv(f"{DATA_DIR}/german.data", sep=" ", header=None, names=cols)
    y = (df["target"] == 1).astype(int)                           # favorable: good credit risk
    prot = {"age": (df["age"] >= 25).astype(int),
            "sex": (~df["personal_status_sex"].isin(["A92", "A95"])).astype(int)}   # A92/A95 = female
    X = df.drop(columns=["target", "age", "personal_status_sex"])
    return X, y, prot

def load_compas():
    df = pd.read_csv(f"{DATA_DIR}/cox-violent-parsed_filt.csv")
    df["c_charge_degree"] = df["c_charge_degree"].str.extract(r"\(([FM])")[0].fillna("O")
    df = df[df["days_b_screening_arrest"].between(-30, 30) & (df["is_recid"] != -1)
            & (df["c_charge_degree"] != "O")]                     # standard ProPublica filters
    df = df.drop_duplicates(subset="id").reset_index(drop=True)   # one row per person
    y = (df["is_recid"] == 0).astype(int)                         # favorable: no recidivism
    prot = {"race": (df["race"] == "Caucasian").astype(int),
            "sex": (df["sex"] == "Female").astype(int)}
    X = df[["age", "juv_fel_count", "juv_misd_count", "juv_other_count", "priors_count", "c_charge_degree"]]
    return X, y, prot

LOADERS = {"Adult": load_adult, "German": load_german, "COMPAS": load_compas}

## Compute the metrics

In [4]:
rows = []
for name, loader in LOADERS.items():
    X, y, prot = loader()
    cons = consistency(make_features(X), y.values)
    for attr, priv in prot.items():
        yv, pv = y.values, priv.values
        rows.append({"Dataset": name, "Protected": attr, "N": len(yv),
                     "N priv": int((pv == 1).sum()), "N unpriv": int((pv == 0).sum()),
                     "P(Y=1|priv)": yv[pv == 1].mean(), "P(Y=1|unpriv)": yv[pv == 0].mean(),
                     "DP_Diff": dp_diff(yv, pv), "DIR": dir_ratio(yv, pv), "Consistency": cons})
results = pd.DataFrame(rows).set_index(["Dataset", "Protected"])
results.round(3)

N  N priv  N unpriv  P(Y=1|priv)  P(Y=1|unpriv)  DP_Diff    DIR  Consistency
Dataset Protected                                                                                  
Adult   sex        32561   21790     10771        0.306          0.109   -0.196  0.358        0.793
        race       32561   27816      4745        0.256          0.153   -0.103  0.596        0.793
German  age         1000     851       149        0.719          0.591   -0.129  0.821        0.672
        sex         1000     690       310        0.723          0.648   -0.075  0.897        0.672
COMPAS  race        8891    2969      5922        0.573          0.485   -0.088  0.847        0.648
        sex         8891    1589      7302        0.656          0.483   -0.173  0.736        0.648

**Reading the table**

* `DP_Diff < 0` and `DIR < 0.8` mean the unprivileged group receives the favorable label clearly less often in the original data, i.e. the data itself already shows disparate impact before any classifier is trained.
* Consistency does not depend on the protected attribute (it uses all remaining features), so it is the same for both rows of a dataset.

In [5]:
results["80% rule violated"] = results["DIR"] < 0.8
results[["DP_Diff", "DIR", "Consistency", "80% rule violated"]].round(3)

DP_Diff    DIR  Consistency  80% rule violated
Dataset Protected                                                
Adult   sex         -0.196  0.358        0.793               True
        race        -0.103  0.596        0.793               True
German  age         -0.129  0.821        0.672              False
        sex         -0.075  0.897        0.672              False
COMPAS  race        -0.088  0.847        0.648              False
        sex         -0.173  0.736        0.648               True

# Task 3 – Stratified split, mitigation pipeline and evaluation on the test set

**Method:** Disparate Impact Remover (Feldman et al., 2015), a **pre-processing** method.

Pipeline for every dataset and protected attribute:
1. Stratified train/test split (stratified by label × protected group).
2. Repair the features of the training data with the Disparate Impact Remover and train a logistic regression on them. The test features are repaired too, then the classifier predicts the final labels on the held-out test set.
3. On the test set, report DP_Diff, DIR, average equalized-odds difference (AEO_Diff), predictive parity difference (PP_Diff), and consistency from the final predictions. The same classifier without mitigation is shown as a baseline.

`NUMERIC_ONLY = True` keeps only numeric features (closer to the paper, which drops unordered categorical features); with `False` all categorical features are one-hot encoded and repaired as well.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import DisparateImpactRemover

SEED = 42
REPAIR_LEVEL = 1.0      # lambda: 0 = no repair, 1 = full repair
NUMERIC_ONLY = False    # True: use numeric features only (closer to the paper, which drops unordered categoricals)

def to_aif(F, y, s):
    d = F.copy()
    d["prot"] = s.values
    d["label"] = y.values
    return BinaryLabelDataset(df=d, label_names=["label"], protected_attribute_names=["prot"],
                              favorable_label=1, unfavorable_label=0)

def repair(F, y, s):
    """Apply the Disparate Impact Remover and return the repaired features (protected column dropped)."""
    remover = DisparateImpactRemover(repair_level=REPAIR_LEVEL, sensitive_attribute="prot")
    ds = remover.fit_transform(to_aif(F, y, s))
    return pd.DataFrame(ds.features, columns=ds.feature_names).drop(columns=["prot"])

def aeo_diff(y_true, y_pred, priv):
    """Average equalized-odds difference: mean of the TPR gap and the FPR gap (unprivileged - privileged)."""
    def rate(group, label):                                  # P(y_pred = 1 | group, y_true = label)
        return y_pred[(priv == group) & (y_true == label)].mean()
    return ((rate(0, 1) - rate(1, 1)) + (rate(0, 0) - rate(1, 0))) / 2

def pp_diff(y_true, y_pred, priv):
    """Predictive parity difference: P(Y=1 | y_pred=1, unprivileged) - P(Y=1 | y_pred=1, privileged)."""
    def precision(group):
        return y_true[(priv == group) & (y_pred == 1)].mean()
    return precision(0) - precision(1)

def evaluate(dataset, attr, model, pred, F_te, y_te, s_te):
    pred = np.asarray(pred); yt, s = np.asarray(y_te), s_te.values
    return {"Dataset": dataset, "Protected": attr, "Model": model,
            "DP_Diff": dp_diff(pred, s_te.values), "DIR": dir_ratio(pred, s_te.values),
            "AEO_Diff": aeo_diff(yt, pred, s), "PP_Diff": pp_diff(yt, pred, s),
            "Consistency": consistency(F_te, pred)}          # computed on the original test features

def run_pipeline(dataset, attr):
    X, y, prot = LOADERS[dataset]()
    s = prot[attr]
    F = make_features(X.select_dtypes("number") if NUMERIC_ONLY else X)

    # 1) stratified split (label x protected group)
    strat = y.astype(str) + "_" + s.astype(str)
    idx_tr, idx_te = train_test_split(F.index, test_size=0.3, stratify=strat, random_state=SEED)
    F_tr, F_te, y_tr, y_te, s_tr, s_te = F.loc[idx_tr], F.loc[idx_te], y.loc[idx_tr], y.loc[idx_te], s.loc[idx_tr], s.loc[idx_te]

    # 2) pre-processing: repair train (and test) features, train the classifier on repaired train data
    clf = LogisticRegression(max_iter=2000, random_state=SEED).fit(repair(F_tr, y_tr, s_tr), y_tr)
    pred_mit = clf.predict(repair(F_te, y_te, s_te))          # final predictions on the held-out test set

    # baseline for comparison: same classifier without mitigation
    pred_base = LogisticRegression(max_iter=2000, random_state=SEED).fit(F_tr, y_tr).predict(F_te)

    # 3) metrics on the held-out test set
    return [evaluate(dataset, attr, "No mitigation", pred_base, F_te, y_te, s_te),
            evaluate(dataset, attr, f"DI remover (λ={REPAIR_LEVEL})", pred_mit, F_te, y_te, s_te)]

pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


pip install 'aif360[inFairness]'


pip install 'aif360[Reductions]'


## Run on all datasets

In [7]:
rows = []
for dataset in LOADERS:
    for attr in LOADERS[dataset]()[2]:        # protected attributes of this dataset
        rows += run_pipeline(dataset, attr)

final = pd.DataFrame(rows).set_index(["Dataset", "Protected", "Model"])
final.round(3)

DP_Diff    DIR  AEO_Diff  PP_Diff  Consistency
Dataset Protected Model                                                             
Adult   sex       No mitigation        -0.170  0.322    -0.093   -0.082        0.942
                  DI remover (λ=1.0)   -0.163  0.331    -0.086   -0.090        0.944
        race      No mitigation        -0.092  0.558    -0.047   -0.027        0.944
                  DI remover (λ=1.0)   -0.074  0.634    -0.017   -0.058        0.945
German  age       No mitigation        -0.228  0.705    -0.148   -0.198        0.779
                  DI remover (λ=1.0)   -0.039  0.957     0.011   -0.170        0.880
        sex       No mitigation        -0.000  1.000     0.028   -0.060        0.780
                  DI remover (λ=1.0)   -0.056  0.934    -0.046   -0.032        0.797
COMPAS  race      No mitigation        -0.181  0.756    -0.158   -0.020        0.984
                  DI remover (λ=1.0)   -0.134  0.809    -0.110   -0.030        0.962
        sex       No mitigation        -0.184  0.748    -0.145   -0.091        0.985
                  DI remover (λ=1.0)   -0.066  0.894    -0.024   -0.113        0.952

## Sensitivity analysis: repair level λ

The only parameter of the Disparate Impact Remover is the repair level λ (0 = original data, 1 = full repair). The pipeline is repeated for several values of λ; accuracy is added to show the fairness/utility trade-off.

In [8]:
from sklearn.metrics import accuracy_score

def run_sensitivity(dataset, attr):
    """Same pipeline as run_pipeline (repaired model only), with accuracy added."""
    X, y, prot = LOADERS[dataset]()
    s = prot[attr]
    F = make_features(X.select_dtypes("number") if NUMERIC_ONLY else X)
    strat = y.astype(str) + "_" + s.astype(str)
    idx_tr, idx_te = train_test_split(F.index, test_size=0.3, stratify=strat, random_state=SEED)
    F_tr, F_te, y_tr, y_te, s_tr, s_te = F.loc[idx_tr], F.loc[idx_te], y.loc[idx_tr], y.loc[idx_te], s.loc[idx_tr], s.loc[idx_te]
    clf = LogisticRegression(max_iter=2000, random_state=SEED).fit(repair(F_tr, y_tr, s_tr), y_tr)
    pred = clf.predict(repair(F_te, y_te, s_te))
    row = evaluate(dataset, attr, f"lambda={REPAIR_LEVEL}", pred, F_te, y_te, s_te)
    row["Accuracy"] = accuracy_score(y_te, pred)
    return row

lambdas = [0, 0.25, 0.5, 0.75, 1.0]
sweep = []
for lam in lambdas:
    REPAIR_LEVEL = lam                           # repair() reads this global value
    for dataset in LOADERS:
        for attr in LOADERS[dataset]()[2]:
            row = run_sensitivity(dataset, attr)
            row["lambda"] = lam
            sweep.append(row)
REPAIR_LEVEL = 1.0                               # restore the default

sweep = pd.DataFrame(sweep)
sweep.pivot_table(index=["Dataset", "Protected"], columns="lambda",
                  values=["DIR", "AEO_Diff", "Accuracy"]).round(3)

AEO_Diff                             Accuracy                                DIR                            
lambda                0.00   0.25   0.50   0.75   1.00     0.00   0.25   0.50   0.75   1.00   0.00   0.25   0.50   0.75   1.00
Dataset Protected                                                                                                             
Adult   race        -0.047 -0.040 -0.034 -0.023 -0.017    0.849  0.850  0.850  0.848  0.847  0.558  0.571  0.582  0.610  0.634
        sex         -0.093 -0.092 -0.090 -0.087 -0.086    0.844  0.843  0.844  0.843  0.843  0.322  0.323  0.327  0.331  0.331
COMPAS  race        -0.158 -0.157 -0.153 -0.110 -0.110    0.660  0.661  0.659  0.663  0.663  0.756  0.757  0.762  0.809  0.809
        sex         -0.145 -0.141 -0.119 -0.028 -0.024    0.660  0.661  0.662  0.656  0.654  0.748  0.748  0.771  0.888  0.894
German  age         -0.148 -0.160 -0.157  0.014  0.011    0.747  0.750  0.747  0.747  0.750  0.705  0.695  0.698  0.961  0.957
        sex          0.028  0.002 -0.007 -0.022 -0.046    0.787  0.777  0.773  0.777  0.777  1.000  0.982  0.976  0.963  0.934